# Eksperimen Awal SOTA YOLO (Colab GPU Edition)
Jalankan notebook ini di Google Colab menggunakan **T4 GPU** untuk melihat performa kecepatan akselerator GPU dalam menangani model *real-time* YOLO di kerumunan.

In [3]:
!pip install ultralytics opencv-python matplotlib

### ⚠️ Persiapan Gambar Uji
Blok di bawah ini akan memunculkan tombol **Upload**. Silakan klik dan pilih gambar teater anak-anak (`273275,e99d80007220d4b6.jpg`) dari laptop Anda.

In [ ]:
import os
import cv2
from ultralytics import YOLO
import matplotlib.pyplot as plt

try:
    from google.colab import files
    print("Silakan klik tombol 'Choose Files' di bawah ini untuk mengupload gambar:\n")
    uploaded = files.upload()
    test_image = list(uploaded.keys())[0]
    print(f"\n✅ Gambar '{test_image}' berhasil di-upload ke server Google!")
except ImportError:
    # Fallback otomatis jika dijalankan di PC Lokal (VSCode biasa)
    test_image = r"g:\semester 6\hibah-riset\models\Images\273275,e99d80007220d4b6.jpg"
    if os.path.exists(test_image):
        print(f"✅ Menjalankan dari lokal. Gambar uji ditemukan: {test_image}")
    else:
        print("❌ Error: Gambar tidak ditemukan! Pastikan path benar.")

Silakan klik tombol 'Choose Files' di bawah ini untuk mengupload gambar:



### 1. Evaluasi Zero-Shot (Menguji Kelemahan Bawaan Pabrik)

In [ ]:
models_to_test = ['yolov10n.pt', 'yolo11n.pt', 'yolo26n.pt']
for model_name in models_to_test:
    print(f'\n--- Menguji Zero-Shot: {model_name} ---')
    model = YOLO(model_name)
    results = model(test_image, classes=[0], conf=0.25)
    res_plotted = results[0].plot()
    
    plt.figure(figsize=(10,8))
    plt.imshow(cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB))
    plt.title(f'{model_name} - Mendeteksi {len(results[0].boxes)} orang (Dari total puluhan)')
    plt.axis('off')
    plt.show()

### 2. Validasi NMS Overhead di GPU
Mengukur apakah di GPU, proses *Non-Maximum Suppression* (NMS) YOLO lama tetap memakan resource waktu lebih besar dari model SOTA NMS-Free (YOLO26).

In [ ]:
print(f"{'Model':<15} | {'Inference (ms)':<15} | {'Post-Process/NMS (ms)':<22} | {'Sifat Model'}")
print("-" * 75)
for model_name in models_to_test:
    model = YOLO(model_name)
    for _ in range(5): model(test_image, verbose=False)
    
    avg_inf, avg_post, iters = 0, 0, 20
    for _ in range(iters):
        results = model(test_image, verbose=False)
        avg_inf += results[0].speed['inference']
        avg_post += results[0].speed['postprocess']
        
    sifat = 'Ada NMS (Lambat)' if '11' in model_name or '8' in model_name else 'NMS-Free (Cepat)'
    print(f"{model_name:<15} | {avg_inf/iters:<15.2f} | {avg_post/iters:<22.2f} | {sifat}")

### 3. Resolution Scaling (Simulasi Edge Limit di Cloud)
Memetakan kemampuan model saat gambarnya dikecilkan secara ekstrim.

In [ ]:
resolutions = [256, 320, 416, 480, 512, 640]
fps_results = {m: [] for m in models_to_test}
det_results = {m: [] for m in models_to_test}

for model_name in models_to_test:
    model = YOLO(model_name)
    for _ in range(3): model(test_image, imgsz=640, verbose=False)
    
    for res in resolutions:
        total_time = 0
        for _ in range(5):
            results = model(test_image, imgsz=res, conf=0.25, verbose=False)
            total_time += results[0].speed['inference'] + results[0].speed['preprocess'] + results[0].speed['postprocess']
        
        fps = 1000.0 / (total_time / 5)
        det_count = len(model(test_image, imgsz=res, conf=0.25, verbose=False)[0].boxes)
        
        fps_results[model_name].append(fps)
        det_results[model_name].append(det_count)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors, markers, labels = ['tab:blue', 'tab:orange', 'tab:green'], ['o', 's', '^'], ['YOLOv10n', 'YOLO11n', 'YOLO26n']

for i, m in enumerate(models_to_test):
    ax1.plot(resolutions, fps_results[m], marker=markers[i], color=colors[i], label=labels[i])
    ax2.plot(resolutions, det_results[m], marker=markers[i], color=colors[i], label=labels[i])

ax1.set_title('Kecepatan GPU (FPS)')
ax1.set_xlabel('Resolusi')
ax1.axhline(y=30, color='red', linestyle='--')
ax1.legend()
ax2.set_title('Akurasi Kerumunan')
ax2.set_xlabel('Resolusi')
ax2.legend()
plt.show()